# Chit 4 baseline: a camera trap that wakes up for animals · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a small CNN on 96 × 96 **grayscale** images (animal vs. empty), measure animal recall when at most **5%** of the empties are kept, in fp32 and int8, and count the cost on a microcontroller.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data
The loader stores 96 × 96 RGB; this baseline asks for grayscale (`grayscale=True`). Colour and resolution are levers you can change.

In [ ]:
from capstone.data.karoo import load, recall_at_empties_kept
kw = dict(grayscale=True, size=96)
X_tr, y_tr = load("train", max_items=2_000 if FAST else None, **kw)
X_va, y_va = load("val", **kw); X_te, y_te = load("test", **kw)
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te)):
    print(f"{name:<5} {len(y):>6,} images, {np.mean(y):.1%} animal")
XTR, XVA, XTE = [X.astype(np.float32) / 255 for X in (X_tr, X_va, X_te)]

## 2. The model, costed before training
Four strided 3 × 3 convolutions (96 → 6 pixels), global average pooling, a linear layer.

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))
p = profile(model, (1, 96, 96))
print(f"params {p['params']:,} | MACs {p['macs']:,} | int8 weights {p['weight_bytes']:,} B | peak act {p['peak_act_bytes']:,} B")
TARGET = dict(max_weight_bytes=1024 * 1024, max_act_bytes=256 * 1024, mac_rate=100e6, max_latency_s=1.0)   # DRAFT values from chits.md
fits(p, TARGET);

## 3. Train (class-weighted) and measure

In [ ]:
w = len(y_tr) / (2 * np.bincount(y_tr, minlength=2))
fit(model, XTR, y_tr, epochs=1 if FAST else 12, bs=64, lr=3e-3, class_weight=w)
prob = torch.softmax(torch.as_tensor(predict(model, XTE)), 1)[:, 1].numpy()
REC32 = recall_at_empties_kept(prob, y_te)
print(f"test animal recall with 5% of empties kept, fp32: {REC32:.4f}")

## 4. int8 (weights and activations), with ONNX Runtime static quantization

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/karoo_fp32.onnx")
ort_int8("my_models/karoo_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/karoo_int8.onnx")
sess8 = ort_session("my_models/karoo_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

P8 = torch.softmax(torch.as_tensor(OUT8), 1)[:, 1].numpy()
REC8 = recall_at_empties_kept(P8, y_te)
print(f"test animal recall with 5% of empties kept, int8: {REC8:.4f}   <- the chit's metric")
print()
print("other operating points   empties kept   recall fp32   recall int8")
for k in (0.20, 0.10, 0.05, 0.02, 0.01):
    print(f"{'':<24} {k:>12.0%} {recall_at_empties_kept(prob, y_te, kept=k):>13.3f} {recall_at_empties_kept(P8, y_te, kept=k):>13.3f}")

In [ ]:
ROW = report("4 karoo", (len(XTR), len(XVA), len(XTE)), (1, 96, 96), f"recall@5%empties int8 {REC8:.4f} (fp32 {REC32:.4f}); @20% {recall_at_empties_kept(P8, y_te, kept=0.20):.3f}, @10% {recall_at_empties_kept(P8, y_te, kept=0.10):.3f}, @2% {recall_at_empties_kept(P8, y_te, kept=0.02):.3f}", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms at 100 M MAC/s",
             "CDLA-Permissive 1.0 (LILA, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.